# HR Chat Agent Evaluation

This notebook implements the four tasks in `interview_test_questions.md`:

1. Import the reference and answer data.
2. Evaluate the quality of `answer_df["agent_answers"]` (the free-text answers).
3. Evaluate the quality of `answer_df["cited_articles"]` (citations shown to the user).
4. Evaluate the quality of `answer_df["search_results"]` (the ranked retrieval step).

For the free-text answer quality (task 2), a plain string-similarity metric is not
enough to judge legal correctness, completeness of thresholds/numbers, or
hallucination. So we use an LLM-as-judge, calling the **Groq API** and grounding the judge in the
`gold_standard_text` for each question.




In [ ]:
import os
import json
import time
from textwrap import dedent

import numpy as np
import pandas as pd


GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "xxxxx")
GROQ_MODEL = os.environ.get("GROQ_MODEL", "meta-llama/llama-4-scout-17b-16e-instruct")

try:
    from groq import Groq
    _groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
except ImportError:
    _groq_client = None
    print("groq package not installed. Run: pip install groq")

if not GROQ_API_KEY:
    print(
        "WARNING: GROQ_API_KEY is not set."
    )



## 1. Import the data

In [29]:
ANSWER_DATA_PATH = "fake_interview_answer_data.json"
REFERENCE_DATA_PATH = "fake_interview_reference_data.json"

answer_df = pd.read_json(ANSWER_DATA_PATH, orient="table")
reference_df = pd.read_json(REFERENCE_DATA_PATH, orient="table")

# Join once up front -- every later evaluation needs both sides.
eval_df = answer_df.join(reference_df, how="inner", lsuffix="_agent", rsuffix="_ref")

print(f"answer_df: {answer_df.shape}, reference_df: {reference_df.shape}, joined: {eval_df.shape}")
eval_df.head()


answer_df: (10, 3), reference_df: (10, 2), joined: (10, 5)


,cited_articles,agent_answers,search_results,optimal_search_results,gold_standard_text
Question,,,,,
What are the key requirements for FMLA eligibility?,"[3875, 4490]","To be eligible for FMLA, an employee must have worked for the employer for at least 6 months, have 1,250 hours of se...","[3875, 1001, 4490, 1022, 1100, 1200, 1300, 1400, 1098, 1600]","[1098, 1042]","To be eligible for FMLA leave, an employee must have worked for the employer for at least 12 months, have at least 1..."
How does the ADA interactive process work?,[421],The ADA interactive process is a formal negotiation where the employer must provide the exact accommodation requeste...,"[587, 200, 421, 201, 202, 203, 205, 206, 207, 208]",[587],The ADA interactive process is a collaborative dialogue between the employer and an employee with a disability to id...
What is the difference between exempt and non-exempt employees under FLSA?,"[2201, 2215, 2203]",Exempt employees are generally paid on a salary basis and are exempt from minimum wage and overtime pay requirements...,"[2201, 3001, 2215, 3002, 2203, 3003, 3004, 3005, 3006, 3007]","[2215, 2201, 2203]","Under the FLSA, exempt employees are not entitled to overtime pay and are generally paid on a salary basis meeting m..."
What are an employer's obligations under WARN Act before a mass layoff?,"[2845, 5332]","Under the federal WARN Act, covered employers must provide a minimum of 30 days of advance written notice to affecte...","[4001, 2845, 4002, 5332, 4003, 4004, 4005, 4006, 4007, 4008]","[1893, 1901]",The Worker Adjustment and Retraining Notification (WARN) Act requires employers with 100 or more full-time employees...
How should employers handle religious accommodation requests?,[765],"Employers must accommodate religious requests unless it causes an 'undue hardship,' which the courts have defined as...","[5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 765, 5009]",[734],"Under Title VII, employers must reasonably accommodate an employee's sincerely held religious beliefs, practices, or..."


## 2. Evaluate the quality of the answer text

We ask an LLM judge (Groq) to compare each `agent_answers` entry against its
`gold_standard_text`, grounded strictly in the reference text (not the model's own
legal knowledge, since employment law details like numeric thresholds change and
we want the judge scoring against *this* rubric, not its training data). The judge
returns structured JSON so we can aggregate scores programmatically.

Rubric dimensions:
- **correctness** (1-5): are the stated facts/thresholds/numbers consistent with the gold standard?
- **completeness** (1-5): does it cover the key points in the gold standard, or omit important nuances?
- **hallucinated** (bool) + **hallucinated_claims**: any specific claims not supported by, or contradicting, the gold standard.
- **missing_key_points**: important gold-standard content the answer left out.


In [18]:
JUDGE_SYSTEM_PROMPT = dedent('''
You are a meticulous legal-content QA reviewer for an HR compliance chatbot.
You will be given a user QUESTION, a GOLD_STANDARD reference answer written by
subject-matter experts, and an AGENT_ANSWER produced by an AI chatbot.

Score the AGENT_ANSWER strictly against the GOLD_STANDARD (not your own general
knowledge, since laws/thresholds change over time and the GOLD_STANDARD is the
source of truth for this evaluation).

Return ONLY a JSON object (no markdown fences, no prose) with this exact schema:
{
  "correctness_score": <int 1-5, 5=fully accurate, 1=materially wrong>,
  "completeness_score": <int 1-5, 5=covers all key points, 1=misses almost everything>,
  "hallucinated": <true|false>,
  "hallucinated_claims": [<short strings describing any claim in AGENT_ANSWER that
       is not supported by, or contradicts, GOLD_STANDARD>],
  "missing_key_points": [<short strings describing important GOLD_STANDARD content
       omitted from AGENT_ANSWER>],
  "explanation": "<1-3 sentence justification>"
}
''').strip()

JUDGE_USER_TEMPLATE = dedent('''
QUESTION:
{question}

GOLD_STANDARD:
{gold_standard}

AGENT_ANSWER:
{agent_answer}
''').strip()


def call_groq_judge(question, gold_standard, agent_answer, retries=2):
    """Calls the Groq chat completions API to score one answer.
    Returns a dict on success, or None if the API is unavailable/fails
    (so the rest of the notebook still runs without a key)."""
    if _groq_client is None:
        return None

    user_prompt = JUDGE_USER_TEMPLATE.format(
        question=question, gold_standard=gold_standard, agent_answer=agent_answer
    )

    for attempt in range(retries + 1):
        try:
            resp = _groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0,
                max_tokens=800,
                response_format={"type": "json_object"},
            )
            content = resp.choices[0].message.content
            return json.loads(content)
        except Exception as e:
            if attempt == retries:
                print(f"Groq judge call failed after {retries + 1} attempts: {e}")
                return None
            time.sleep(1.5 * (attempt + 1))
    return None


In [19]:
judge_results = []
for question, row in eval_df.iterrows():
    result = call_groq_judge(
        question=question,
        gold_standard=row["gold_standard_text"],
        agent_answer=row["agent_answers"],
    )
    judge_results.append(result)

judge_df = pd.DataFrame(judge_results, index=eval_df.index)
judge_df.head(10)


,correctness_score,completeness_score,hallucinated,hallucinated_claims,missing_key_points,explanation
Question,,,,,,
What are the key requirements for FMLA eligibility?,1,1,True,"[6 months of employment instead of 12 months, 100-mile radius instead of 75 miles]","[12 months of employment do not need to be consecutive, employment prior to a break in service of seven years or mor...",AGENT_ANSWER materially misstates FMLA eligibility requirements
How does the ADA interactive process work?,1,1,True,"[employer must provide exact accommodation requested by employee, requirement of a signed doctor's note]","[collaborative dialogue, identifying essential functions, exploring potential accommodations]","AGENT_ANSWER is materially wrong and incomplete, introducing new requirements not present in GOLD_STANDARD."
What is the difference between exempt and non-exempt employees under FLSA?,4,3,False,[],"[duties tests for exemptions, minimum thresholds for salary basis, overtime pay at 1.5 times regular rate for non-ex...",The AGENT_ANSWER correctly identifies the basic distinction between exempt and non-exempt employees but omits crucia...
What are an employer's obligations under WARN Act before a mass layoff?,2,2,True,"[The WARN Act requires 30 days of advance written notice, which is less than the required 60 calendar days]","[Notice must be provided to affected workers or their representatives, the state dislocated worker unit, and the loc...",The AGENT_ANSWER incorrectly states the required notice period as 30 days instead of 60 calendar days and omits key ...
How should employers handle religious accommodation requests?,1,1,True,[The definition of 'undue hardship' is still 'more than a de minimis cost'],"[The Supreme Court's decision in Groff v. DeJoy (2023) changed the definition of undue hardship, Examples of common ...",The AGENT_ANSWER uses an outdated definition of 'undue hardship' and omits crucial information from the GOLD_STANDARD.
What constitutes a hostile work environment under Title VII?,4,4,False,[],"[totality of circumstances including frequency, severity, physically threatening or humiliating, and interference wi...",The AGENT_ANSWER accurately captures the essence of a hostile work environment under Title VII but lacks some detail...
What are the recordkeeping requirements for I-9 forms?,2,1,True,[retention period of 5 years],"[availability for inspection, electronic storage requirements, penalties for violations, storage separate from perso...","The AGENT_ANSWER incorrectly states a 5-year retention period, contradicting the GOLD_STANDARD's 3-year requirement...."
How does COBRA continuation coverage work after termination?,1,1,True,"[employers with 15 or more employees, up to 12 months, employer is required to pay at least 50% of the premium]","[20 or more employees, 18 months for termination, 36 months for other qualifying events, up to 102% of the full prem...",AGENT_ANSWER contains multiple inaccuracies and omissions compared to GOLD_STANDARD.
What are the rules around compensable travel time for non-exempt employees?,3,2,False,[],"[compensable travel time during normal work hours, overnight travel rules]",The AGENT_ANSWER correctly identifies that ordinary commuting is not compensable and travel between work sites durin...


In [20]:
if judge_df.notna().any(axis=None):
    summary = pd.DataFrame({
        "mean_correctness": [judge_df["correctness_score"].mean()],
        "mean_completeness": [judge_df["completeness_score"].mean()],
        "pct_hallucinated": [judge_df["hallucinated"].mean() * 100 if "hallucinated" in judge_df else np.nan],
        "n_scored": [judge_df["correctness_score"].notna().sum()],
        "n_total": [len(judge_df)],
    })
    print("Answer-quality summary (LLM judge, Groq):")
    display(summary)

    print("\nLowest-scoring answers (correctness + completeness):")
    ranked = judge_df.copy()
    ranked["combined"] = ranked["correctness_score"].fillna(0) + ranked["completeness_score"].fillna(0)
    display(ranked.sort_values("combined").head(5)[
        ["correctness_score", "completeness_score", "hallucinated", "hallucinated_claims", "missing_key_points", "explanation"]
    ])
else:
    print("No LLM-judge scores available (GROQ_API_KEY not set or all calls failed). "
          "Set GROQ_API_KEY and re-run this section to get correctness/completeness/hallucination scores.")


Answer-quality summary (LLM judge, Groq):


,mean_correctness,mean_completeness,pct_hallucinated,n_scored,n_total
0,2.3,1.9,60.0,10,10



Lowest-scoring answers (correctness + completeness):


,correctness_score,completeness_score,hallucinated,hallucinated_claims,missing_key_points,explanation
Question,,,,,,
What are the key requirements for FMLA eligibility?,1,1,True,"[6 months of employment instead of 12 months, 100-mile radius instead of 75 miles]","[12 months of employment do not need to be consecutive, employment prior to a break in service of seven years or mor...",AGENT_ANSWER materially misstates FMLA eligibility requirements
How does the ADA interactive process work?,1,1,True,"[employer must provide exact accommodation requested by employee, requirement of a signed doctor's note]","[collaborative dialogue, identifying essential functions, exploring potential accommodations]","AGENT_ANSWER is materially wrong and incomplete, introducing new requirements not present in GOLD_STANDARD."
How does COBRA continuation coverage work after termination?,1,1,True,"[employers with 15 or more employees, up to 12 months, employer is required to pay at least 50% of the premium]","[20 or more employees, 18 months for termination, 36 months for other qualifying events, up to 102% of the full prem...",AGENT_ANSWER contains multiple inaccuracies and omissions compared to GOLD_STANDARD.
How should employers handle religious accommodation requests?,1,1,True,[The definition of 'undue hardship' is still 'more than a de minimis cost'],"[The Supreme Court's decision in Groff v. DeJoy (2023) changed the definition of undue hardship, Examples of common ...",The AGENT_ANSWER uses an outdated definition of 'undue hardship' and omits crucial information from the GOLD_STANDARD.
What are the recordkeeping requirements for I-9 forms?,2,1,True,[retention period of 5 years],"[availability for inspection, electronic storage requirements, penalties for violations, storage separate from perso...","The AGENT_ANSWER incorrectly states a 5-year retention period, contradicting the GOLD_STANDARD's 3-year requirement...."


## 3. Evaluate the quality of the cited articles

`cited_articles` are the sources the agent showed the end user as backing its
answer. We compare them against `optimal_search_results` (the editorially
verified relevant articles) using standard set-based precision/recall/F1, per
question and in aggregate (micro-averaged across all citations, which weights
questions with more ground-truth articles proportionally more).

We also check **citation grounding**: whether every cited article actually came
from the agent's own `search_results` (i.e. the agent isn't citing an article it
never retrieved -- a specific, cheap-to-detect hallucination-of-source failure
mode that's distinct from the free-text hallucination the LLM judge checks for).


In [22]:
def set_prf1(predicted, actual):
    predicted, actual = set(predicted), set(actual)
    tp = len(predicted & actual)
    precision = tp / len(predicted) if predicted else np.nan  # undefined only if agent cited nothing
    recall = tp / len(actual) if actual else np.nan  # undefined only if no ground-truth articles exist
    if pd.isna(precision) or pd.isna(recall):
        f1 = np.nan
    elif (precision + recall) == 0:
        f1 = 0.0  # correctly score "zero overlap" as 0, not NaN (0 is falsy but not undefined)
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

citation_rows = []
for question, row in eval_df.iterrows():
    cited = row["cited_articles"]
    optimal = row["optimal_search_results"]
    retrieved = row["search_results"]

    precision, recall, f1 = set_prf1(cited, optimal)
    ungrounded = [c for c in cited if c not in set(retrieved)]  # cited but never retrieved

    citation_rows.append({
        "question": question,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_cited": len(cited),
        "n_optimal": len(optimal),
        "ungrounded_citations": ungrounded,
    })

citation_df = pd.DataFrame(citation_rows).set_index("question")
citation_df


,precision,recall,f1,n_cited,n_optimal,ungrounded_citations
question,,,,,,
What are the key requirements for FMLA eligibility?,0.0,0.0,0.0,2,2,[]
How does the ADA interactive process work?,0.0,0.0,0.0,1,1,[]
What is the difference between exempt and non-exempt employees under FLSA?,1.0,1.0,1.0,3,3,[]
What are an employer's obligations under WARN Act before a mass layoff?,0.0,0.0,0.0,2,2,[]
How should employers handle religious accommodation requests?,0.0,0.0,0.0,1,1,[]
What constitutes a hostile work environment under Title VII?,1.0,1.0,1.0,2,2,[]
What are the recordkeeping requirements for I-9 forms?,1.0,1.0,1.0,1,1,[]
How does COBRA continuation coverage work after termination?,0.0,0.0,0.0,3,3,[]
What are the rules around compensable travel time for non-exempt employees?,1.0,1.0,1.0,2,2,[]


In [23]:
# Micro-averaged precision/recall/F1 across all questions (pools all TP/FP/FN)
tp_total = fp_total = fn_total = 0
for question, row in eval_df.iterrows():
    cited, optimal = set(row["cited_articles"]), set(row["optimal_search_results"])
    tp_total += len(cited & optimal)
    fp_total += len(cited - optimal)
    fn_total += len(optimal - cited)

micro_precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else np.nan
micro_recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else np.nan
micro_f1 = (
    2 * micro_precision * micro_recall / (micro_precision + micro_recall)
    if micro_precision and micro_recall else np.nan
)

print("Citation quality summary:")
print(f"  Macro-avg precision: {citation_df['precision'].mean():.3f}")
print(f"  Macro-avg recall:    {citation_df['recall'].mean():.3f}")
print(f"  Macro-avg F1:        {citation_df['f1'].mean():.3f}")
print(f"  Micro-avg precision: {micro_precision:.3f}")
print(f"  Micro-avg recall:    {micro_recall:.3f}")
print(f"  Micro-avg F1:        {micro_f1:.3f}")
print(f"  Questions with an ungrounded citation (cited but never retrieved): "
      f"{(citation_df['ungrounded_citations'].str.len() > 0).sum()} / {len(citation_df)}")


Citation quality summary:
  Macro-avg precision: 0.400
  Macro-avg recall:    0.400
  Macro-avg F1:        0.400
  Micro-avg precision: 0.444
  Micro-avg recall:    0.444
  Micro-avg F1:        0.444
  Questions with an ungrounded citation (cited but never retrieved): 0 / 10


## 4. Evaluate the quality of the search retrieval

`search_results` is a *ranked* list, so unlike the citation check above we use
ranking-aware IR metrics against `optimal_search_results`:

- **Precision@k / Recall@k** -- of the top-k retrieved, how many are relevant / of
  all relevant docs, how many were retrieved in the top-k.
- **MRR** (Mean Reciprocal Rank) -- 1 / rank of the first relevant result; rewards
  surfacing at least one good result early.
- **Average Precision (AP)** -- precision averaged at each rank where a relevant
  doc appears; rewards ranking *all* relevant docs near the top, not just the first.
- **NDCG@k** -- discounts relevant docs by log2(rank+1); the standard ranking-quality
  metric that accounts for position.

We use k=10 by default since every `search_results` list in this dataset has 10 entries.


In [24]:
def retrieval_metrics(search_results, optimal_results, k=10):
    relevant = set(optimal_results)
    ranked = list(search_results)
    top_k = ranked[:k]

    hits_k = [1 if doc in relevant else 0 for doc in top_k]
    precision_at_k = sum(hits_k) / k if k else np.nan
    recall_at_k = sum(hits_k) / len(relevant) if relevant else np.nan

    # Mean Reciprocal Rank (first relevant hit)
    rr = 0.0
    for i, doc in enumerate(ranked):
        if doc in relevant:
            rr = 1.0 / (i + 1)
            break

    # Average Precision
    num_relevant_seen = 0
    precisions_at_hits = []
    for i, doc in enumerate(ranked):
        if doc in relevant:
            num_relevant_seen += 1
            precisions_at_hits.append(num_relevant_seen / (i + 1))
    ap = float(np.mean(precisions_at_hits)) if precisions_at_hits else 0.0

    # NDCG@k (binary relevance)
    dcg = sum(1.0 / np.log2(i + 2) for i, doc in enumerate(top_k) if doc in relevant)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

    return {
        "precision_at_k": precision_at_k,
        "recall_at_k": recall_at_k,
        "mrr": rr,
        "average_precision": ap,
        "ndcg_at_k": ndcg_at_k,
    }

retrieval_rows = []
for question, row in eval_df.iterrows():
    metrics = retrieval_metrics(row["search_results"], row["optimal_search_results"], k=10)
    metrics["question"] = question
    retrieval_rows.append(metrics)

retrieval_df = pd.DataFrame(retrieval_rows).set_index("question")
retrieval_df


,precision_at_k,recall_at_k,mrr,average_precision,ndcg_at_k
question,,,,,
What are the key requirements for FMLA eligibility?,0.1,0.5,0.111111,0.111111,0.184576
How does the ADA interactive process work?,0.1,1.0,1.000000,1.000000,1.000000
What is the difference between exempt and non-exempt employees under FLSA?,0.3,1.0,1.000000,0.755556,0.885460
What are an employer's obligations under WARN Act before a mass layoff?,0.0,0.0,0.000000,0.000000,0.000000
How should employers handle religious accommodation requests?,0.0,0.0,0.000000,0.000000,0.000000
What constitutes a hostile work environment under Title VII?,0.2,1.0,1.000000,0.750000,0.877215
What are the recordkeeping requirements for I-9 forms?,0.1,1.0,0.500000,0.500000,0.630930
How does COBRA continuation coverage work after termination?,0.0,0.0,0.000000,0.000000,0.000000
What are the rules around compensable travel time for non-exempt employees?,0.2,1.0,0.500000,0.500000,0.650921


In [25]:
print("Retrieval quality summary (mean across questions):")
display(retrieval_df.mean(numeric_only=True).to_frame("mean").T)

# Mean Average Precision (MAP) -- the standard single-number IR summary metric
map_score = retrieval_df["average_precision"].mean()
print(f"\nMAP (Mean Average Precision): {map_score:.3f}")
print(f"MRR:                          {retrieval_df['mrr'].mean():.3f}")
print(f"NDCG@10:                      {retrieval_df['ndcg_at_k'].mean():.3f}")


Retrieval quality summary (mean across questions):


,precision_at_k,recall_at_k,mrr,average_precision,ndcg_at_k
mean,0.11,0.65,0.431111,0.381667,0.461595



MAP (Mean Average Precision): 0.382
MRR:                          0.431
NDCG@10:                      0.462


## 5. Putting it together

A quick combined view: for each question, the retrieval quality (did search surface
the right articles?), the citation quality (did the agent then cite the right
ones?), and the answer quality (did the final text get the facts right?). This
makes it easy to see *where in the pipeline* a given question is failing --
e.g. good retrieval + poor citation suggests a citation-selection bug, while poor
retrieval alone explains a downstream wrong answer even if the LLM's writing looks
fluent.


In [26]:
combined = retrieval_df[["ndcg_at_k", "average_precision"]].join(
    citation_df[["precision", "recall", "f1"]], lsuffix="_retrieval", rsuffix="_citation"
)
if judge_df.notna().any(axis=None):
    combined = combined.join(judge_df[["correctness_score", "completeness_score", "hallucinated"]])

combined


,ndcg_at_k,average_precision,precision,recall,f1,correctness_score,completeness_score,hallucinated
question,,,,,,,,
What are the key requirements for FMLA eligibility?,0.184576,0.111111,0.0,0.0,0.0,1,1,True
How does the ADA interactive process work?,1.000000,1.000000,0.0,0.0,0.0,1,1,True
What is the difference between exempt and non-exempt employees under FLSA?,0.885460,0.755556,1.0,1.0,1.0,4,3,False
What are an employer's obligations under WARN Act before a mass layoff?,0.000000,0.000000,0.0,0.0,0.0,2,2,True
How should employers handle religious accommodation requests?,0.000000,0.000000,0.0,0.0,0.0,1,1,True
What constitutes a hostile work environment under Title VII?,0.877215,0.750000,1.0,1.0,1.0,4,4,False
What are the recordkeeping requirements for I-9 forms?,0.630930,0.500000,1.0,1.0,1.0,2,1,True
How does COBRA continuation coverage work after termination?,0.000000,0.000000,0.0,0.0,0.0,1,1,True
What are the rules around compensable travel time for non-exempt employees?,0.650921,0.500000,1.0,1.0,1.0,3,2,False
